# Deep Compression — Pruning, Trained Quantization and Huffman Coding (2015)

_Papers · Efficiency & Compression_

**Shrink a trained network 35x-49x with no accuracy loss by pruning connections, sharing a few weight values, and Huffman-coding them.**

---

This notebook builds the weight-sharing core of Deep Compression from scratch, one step at a time. Run each cell, read the note above it, and watch how a few shared weight values can replace thousands with almost no accuracy loss — then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We reproduce the **weight-sharing** stage of Deep Compression: cluster a layer's weights with k-means and replace every weight by its nearest centroid, so the layer stores just a few shared values plus an index per weight. We do it in four steps: (1) check k-means on a tiny vector by hand, (2) train a small MNIST classifier, (3) write the k-means quantizer, (4) sweep the cluster count and watch accuracy vs bits-per-weight.

### Step 1 — Check 1-D k-means on a tiny weight vector

Before touching a network, we cluster six weights into `k = 2` shared values by hand. We initialise the centroids **linearly** between the min and max weight (the paper's best init), then repeat: assign each weight to its nearest centroid, and move each centroid to the mean of its cluster. We also compute the within-cluster sum of squares (Eqn. 2) and the compression rate `r` (Eqn. 1) for this toy case.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import copy
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)

# --- Sanity-check the worked example: k-means a tiny weight vector by hand (k=2). ---
def kmeans_1d(w, k, iters=25):
    w = torch.as_tensor(w, dtype=torch.float32)
    cent = torch.linspace(w.min(), w.max(), k)        # LINEAR init (the paper's best)
    for _ in range(iters):
        idx = (w[:, None] - cent[None, :]).abs().argmin(1)   # assign to nearest centroid
        for j in range(k):
            sel = w[idx == j]
            if len(sel) > 0:
                cent[j] = sel.mean()                  # update centroid = cluster mean
    return cent, idx

w = [-0.98, -0.95, 0.02, 0.05, 0.91, 1.05]
cent, idx = kmeans_1d(w, 2)

wcss = sum((torch.tensor(w) - cent[idx]) ** 2).item()   # within-cluster sum of squares (Eqn. 2)
n, b, k = 6, 32, 2
r = n * b / (n * np.log2(k) + k * b)                    # Eqn. 1 compression rate
print("centroids   =", [round(c, 4) for c in cent.tolist()])  # [-0.965, 0.5075]
print("assignments =", idx.tolist())                          # [0, 0, 1, 1, 1, 1]
print("WCSS (Eqn.2)=", round(wcss, 4))                        # 0.9037
print("rate r (Eqn.1) =", round(r, 3), "x   bits/weight =", np.log2(k))  # 2.743 x, 1.0

### Step 2 — Train a small MNIST classifier

We need a real trained network whose weights we can then quantize. We load a 6,000-image MNIST subset, flatten each image to 784 values, and train a `784 → 300 → 100 → 10` fully-connected net with Adam for 12 epochs. The `float32` test accuracy printed here is our **baseline** — the number we will try to preserve after weight sharing.

In [ ]:
# --- Train a small fully-connected net on an MNIST subset (torchvision, preinstalled). ---
tfm = T.Compose([T.ToTensor()])
tr = torchvision.datasets.MNIST(root="./data", train=True,  download=True, transform=tfm)
te = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

def stack(ds, m):
    images = torch.stack([ds[i][0].view(-1) for i in range(m)])
    labels = torch.tensor([ds[i][1] for i in range(m)])
    return images, labels

Xtr, ytr = stack(tr, 6000)
Xte, yte = stack(te, 2000)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 300)
        self.fc2 = nn.Linear(300, 100)
        self.fc3 = nn.Linear(100, 10)
    def forward(self, x):
        h = torch.relu(self.fc1(x))
        h = torch.relu(self.fc2(h))
        return self.fc3(h)

net = Net()
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
lf = nn.CrossEntropyLoss()
for ep in range(12):
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 128):
        b_ = perm[i:i + 128]
        opt.zero_grad()
        loss = lf(net(Xtr[b_]), ytr[b_])
        loss.backward()
        opt.step()

def acc(m):
    with torch.no_grad():
        return (m(Xte).argmax(1) == yte).float().mean().item()

base = acc(net)
print("float32 baseline test acc =", round(base, 4))   # ~0.91 (our run, not the paper's)

### Step 3 — Write the weight-sharing quantizer

This is the core operation: take one weight matrix, run the same k-means as Step 1 over all its values (linear init, assign, update), and return a matrix where every weight has been **replaced by its centroid**. After this the matrix contains only `k` distinct values, so it can be stored as `k` floats plus a `log2(k)`-bit index per weight.

In [ ]:
# --- Weight sharing: k-means quantize a weight MATRIX to k shared centroids. ---
def kmeans_quantize(weights, k, iters=20):
    flat = weights.flatten()
    cent = torch.linspace(flat.min(), flat.max(), k)          # linear init
    for _ in range(iters):
        idx = (flat[:, None] - cent[None, :]).abs().argmin(1)  # assign (Eqn. 2)
        for j in range(k):
            sel = flat[idx == j]
            if len(sel) > 0:
                cent[j] = sel.mean()                           # update
    return cent[idx].reshape(weights.shape)                    # replace weight by its centroid

### Step 4 — Sweep the cluster count

Now the ablation. For each `k` we copy the trained net, quantize **every** weight matrix to `k` shared values, and measure test accuracy — changing only the cluster count. `bits/weight` is `log2(k)`. Accuracy stays roughly flat from 256 clusters down to about 32 (5 bits), then drops at 8 and clearly at 2 — showing the trained weights are highly redundant.

In [ ]:
# --- ABLATION: sweep number of clusters k -> accuracy vs bits-per-weight (log2 k). ---
print(" k   bits/weight   test_acc")
for k in [2, 4, 8, 16, 32, 64, 256]:
    qnet = copy.deepcopy(net)
    for name, p in qnet.named_parameters():
        if "weight" in name:                                   # quantize every weight matrix
            p.data = kmeans_quantize(p.data, k)
    print(f"{k:4d}   {int(np.log2(k)):^11d}   {acc(qnet):.4f}")
# Accuracy is ~flat from 256 down to ~32 clusters (5 bits), then drops at 8 and clearly at 2.
# This is OUR small run (quantize-and-stop, no retraining), not the paper's reported number.

## Visualize the data & results

_After k-means weight sharing, how does test accuracy depend on bits-per-weight (= log2 of the number of clusters)?_

### Step 1 — Rebuild and train the baseline net

This cell is self-contained so it can run on its own. We re-import the libraries, reload the same MNIST subset, rebuild the same `784 → 300 → 100 → 10` network, and train it for 12 epochs — reproducing the `float32` baseline from the reference implementation.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import copy
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)

tfm = T.Compose([T.ToTensor()])
tr = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=tfm)
te = torchvision.datasets.MNIST("./data", train=False, download=True, transform=tfm)

def stack(ds, m):
    images = torch.stack([ds[i][0].view(-1) for i in range(m)])
    labels = torch.tensor([ds[i][1] for i in range(m)])
    return images, labels

Xtr, ytr = stack(tr, 6000)
Xte, yte = stack(te, 2000)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 300)
        self.fc2 = nn.Linear(300, 100)
        self.fc3 = nn.Linear(100, 10)
    def forward(self, x):
        h = torch.relu(self.fc1(x))
        h = torch.relu(self.fc2(h))
        return self.fc3(h)

net = Net()
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
lf = nn.CrossEntropyLoss()
for ep in range(12):
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 128):
        b_ = perm[i:i + 128]
        opt.zero_grad()
        lf(net(Xtr[b_]), ytr[b_]).backward()
        opt.step()

def acc(m):
    with torch.no_grad():
        return (m(Xte).argmax(1) == yte).float().mean().item()

base = acc(net)

### Step 2 — Quantize across cluster counts and collect the curve

We reuse the k-means quantizer, then for each `k` we deep-copy the net, quantize every weight matrix, and record `[bits/weight, accuracy]` as one point. The printed list is the accuracy-vs-bits curve: flat down to ~5 bits, then falling — exactly the weight-redundancy story from the reference run.

In [ ]:
def kmeans_quantize(w, k, iters=20):
    f = w.flatten()
    c = torch.linspace(f.min(), f.max(), k)
    for _ in range(iters):
        idx = (f[:, None] - c[None, :]).abs().argmin(1)
        for j in range(k):
            s = f[idx == j]
            if len(s) > 0:
                c[j] = s.mean()
    return c[idx].reshape(w.shape)

pts = []
for k in [2, 4, 8, 16, 32, 64, 256]:
    q = copy.deepcopy(net)
    for n_, p in q.named_parameters():
        if "weight" in n_:
            p.data = kmeans_quantize(p.data, k)
    pts.append([int(np.log2(k)), round(acc(q), 4)])
print("baseline =", round(base, 4))
print("bits, acc =", pts)
# baseline ~0.9145; accuracy flat down to 5 bits, then drops -> our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The cluster-count ablation. You have a trained net at 91% test accuracy. You quantize its
            weight matrices with k-means and sweep $k = 256, 32, 8, 2$ clusters. What do you expect to happen
            to accuracy across that sweep, and what does the result demonstrate about weight redundancy?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Quantize all weight matrices with $k$ centroids (linear init), replace each weight by its centroid, and measure test accuracy &mdash; changing only $k$ each run. — _An honest ablation varies exactly one knob (the cluster count) so any accuracy change is attributable to it._
- Read off the curve: accuracy is essentially unchanged from 256 down to about 32 clusters (5 bits), then dips at 8, and falls clearly at 2 (1 bit). — _5 bits is the paper's reported sweet spot for fully-connected layers; below it, distinct weights are forced to collide and information is lost._
- Conclude the trained weights are highly redundant: a few dozen shared values capture nearly all the useful structure. — _That redundancy is exactly what weight sharing exploits to drop 32 bits per weight down to ~5 with no accuracy loss._

**Answer:** Accuracy stays flat from 256 clusters down to ~32 (5 bits), then degrades as $k$ shrinks
                 further, with a clear drop at 2 clusters. This demonstrates that the trained weights are very
                 redundant &mdash; about 32 shared values are enough &mdash; which is why weight sharing
                 compresses the network with little or no accuracy loss. The CODEVIZ panel shows exactly this
                 accuracy-vs-bits curve from our small run.

</details>

**Problem 2.** A fully-connected layer has $n = 1{,}000{,}000$ weights stored at $b = 32$ bits. You quantize to
            $k = 32$ shared centroids. Using Eqn. 1, what is the compression rate $r$, and how does it compare
            to the rough estimate $b / \log_2 k$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute bits per weight after: $\log_2 32 = 5$. — _Each weight now stores a 5-bit index naming one of 32 centroids instead of a 32-bit float._
- Plug into Eqn. 1: $r = \dfrac{n b}{n \log_2 k + k b} = \dfrac{10^6 \cdot 32}{10^6 \cdot 5 + 32 \cdot 32} = \dfrac{32{,}000{,}000}{5{,}001{,}024} \approx 6.4$. — _The denominator's codebook term $32\cdot32 = 1024$ is negligible against the $5{,}000{,}000$ index bits._
- Compare to $b/\log_2 k = 32/5 = 6.4$. — _When $n \gg k$ the codebook term vanishes and Eqn. 1 collapses to $b/\log_2 k$._

**Answer:** $r \approx 6.4\times$, essentially equal to $b/\log_2 k = 32/5 = 6.4$. With a million weights
                 the 32-entry codebook costs almost nothing, so the compression rate is just the ratio of bits
                 before to bits after per weight. (This is the quantization stage alone; pruning and Huffman
                 multiply it further to the paper's 35x&ndash;49x.)

</details>

**Problem 3.** In the worked example, $w = [-0.98,-0.95,0.02,0.05,0.91,1.05]$ clustered to $k=2$ gave centroids
            $[-0.965, 0.5075]$ with assignments $[0,0,1,1,1,1]$. During retraining, suppose back-propagation
            produces per-weight gradients $g = [0.1, 0.2, -0.3, 0.0, 0.4, -0.1]$. What gradient does centroid
            $C_1$ (index 1) receive (Eqn. 3), and why do all four weights move together?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify which weights belong to cluster 1: indices 2,3,4,5 (assignments equal 1), i.e. weights $0.02, 0.05, 0.91, 1.05$. — _Eqn. 3's indicator $\mathbb{1}(I_{ij}=1)$ selects exactly the weights assigned to centroid 1._
- Sum their gradients: $-0.3 + 0.0 + 0.4 + (-0.1) = 0.0$. — _Eqn. 3 adds the loss gradients of all weights in the cluster to update the one shared value._
- Note $C_1$'s gradient is $0.0$ here, so it would not move this step; all four weights share that single update. — _The four weights are literally the same stored number $C_1$, so the chain rule ties their gradients into one sum._

**Answer:** $C_1$ receives the sum of the gradients of its cluster's weights: $-0.3 + 0.0 + 0.4 - 0.1 =
                 0.0$, so it does not move this step. All four weights move together because after quantization
                 they are one shared value $C_1$; by the chain rule a tied parameter's gradient is the sum of
                 the gradients of every place it is used (Eqn. 3). Centroid $C_0$ would similarly receive
                 $0.1 + 0.2 = 0.3$ from weights 0 and 1.

</details>